In [0]:
%pip install sqlalchemy psycopg2-binary pyarrow #install dep.

In [0]:
import boto3
import pandas as pd
import io
from sqlalchemy import create_engine, text

AWS_ACCESS_KEY = dbutils.secrets.get(scope="aws-credentials", key="access-key")
AWS_SECRET_KEY = dbutils.secrets.get(scope="aws-credentials", key="secret-key")
RDS_HOST       = dbutils.secrets.get(scope="aws-credentials", key="rds-host")
RDS_USER       = dbutils.secrets.get(scope="aws-credentials", key="rds-user")
RDS_PASS       = dbutils.secrets.get(scope="aws-credentials", key="rds-password")
BUCKET         = "enem-data-lake-gblzera"
REGION         = "sa-east-1"

s3 = boto3.client("s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)

engine = create_engine(
    f"postgresql+psycopg2://{RDS_USER}:{RDS_PASS}@{RDS_HOST}:5432/enem_db"
)
print("aqui foi 3")

In [0]:
# Justificativa arquitetural:
# Parquet é o formato colunar padrão para data lakes.
# Compressão snappy reduz ~70% do tamanho vs CSV.

print("Lendo TRU do RDS em chunks e convertendo para Parquet...")

CHUNK_SIZE = 200_000
offset     = 0
frames     = []

while True:
    chunk = pd.read_sql(
        f"SELECT * FROM tru.enem_candidatos LIMIT {CHUNK_SIZE} OFFSET {offset}",
        engine
    )
    if chunk.empty:
        break
    frames.append(chunk)
    offset += CHUNK_SIZE
    print(f"  {offset:,} registros lidos...")

df_tru = pd.concat(frames, ignore_index=True)
print(f"\ntotal carregado: {len(df_tru):,} registros")
print(f"  colunas: {df_tru.shape[1]}")
print(f"  tamanho em memoria: {df_tru.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [0]:
import io

# Converter para Parquet em memória e fazer upload para S3 silver/
buffer = io.BytesIO()
df_tru.to_parquet(buffer, engine="pyarrow", compression="snappy", index=False)
buffer.seek(0)

tamanho_mb = buffer.getbuffer().nbytes / 1024**2
print(f"tamanho do Parquet (snappy): {tamanho_mb:.1f} MB")

s3.put_object(
    Bucket=BUCKET,
    Key="silver/enem/2022/enem_candidatos.parquet",
    Body=buffer.getvalue()
)
print("parquet salvo em s3://enem-data-lake-gblzera/silver/enem/2022/enem_candidatos.parquet")

# Validar lendo de volta
obj        = s3.get_object(Bucket=BUCKET, Key="silver/enem/2022/enem_candidatos.parquet")
df_val     = pd.read_parquet(io.BytesIO(obj["Body"].read()))
print(f"\nvalidação, registros lidos do parquet: {len(df_val):,}")
print(f"  colunas: {df_val.columns.tolist()}")

In [0]:
import pandas as pd

queries = {
    "enem_por_uf": """
        SELECT
            sg_uf_residencia                          AS uf,
            nu_ano,
            COUNT(*)                                  AS total_candidatos,
            COUNT(*) FILTER (WHERE in_treineiro IS FALSE) AS candidatos_regulares,
            ROUND(AVG(nu_nota_cn)::numeric, 2)        AS media_cn,
            ROUND(AVG(nu_nota_ch)::numeric, 2)        AS media_ch,
            ROUND(AVG(nu_nota_lc)::numeric, 2)        AS media_lc,
            ROUND(AVG(nu_nota_mt)::numeric, 2)        AS media_mt,
            ROUND(AVG(nu_nota_redacao)::numeric, 2)   AS media_redacao,
            ROUND(AVG(nu_media_total)::numeric, 2)    AS media_total,
            ROUND(STDDEV(nu_media_total)::numeric, 2) AS desvpad_total
        FROM tru.enem_candidatos
        WHERE sg_uf_residencia IS NOT NULL
        GROUP BY sg_uf_residencia, nu_ano
        ORDER BY media_total DESC
    """,
    "enem_por_municipio": """
        SELECT
            co_municipio_residencia                 AS cod_municipio,
            no_municipio_residencia                 AS municipio,
            sg_uf_residencia                        AS uf,
            nu_ano,
            COUNT(*)                                AS total_candidatos,
            ROUND(AVG(nu_nota_cn)::numeric, 2)      AS media_cn,
            ROUND(AVG(nu_nota_ch)::numeric, 2)      AS media_ch,
            ROUND(AVG(nu_nota_lc)::numeric, 2)      AS media_lc,
            ROUND(AVG(nu_nota_mt)::numeric, 2)      AS media_mt,
            ROUND(AVG(nu_nota_redacao)::numeric, 2) AS media_redacao,
            ROUND(AVG(nu_media_total)::numeric, 2)  AS media_total
        FROM tru.enem_candidatos
        WHERE co_municipio_residencia IS NOT NULL
        GROUP BY co_municipio_residencia, no_municipio_residencia,
                 sg_uf_residencia, nu_ano
        ORDER BY media_total DESC
    """,
    "enem_por_escola": """
        SELECT
            ds_escola,
            sg_uf_residencia                        AS uf,
            nu_ano,
            COUNT(*)                                AS total_candidatos,
            ROUND(AVG(nu_nota_cn)::numeric, 2)      AS media_cn,
            ROUND(AVG(nu_nota_ch)::numeric, 2)      AS media_ch,
            ROUND(AVG(nu_nota_lc)::numeric, 2)      AS media_lc,
            ROUND(AVG(nu_nota_mt)::numeric, 2)      AS media_mt,
            ROUND(AVG(nu_nota_redacao)::numeric, 2) AS media_redacao,
            ROUND(AVG(nu_media_total)::numeric, 2)  AS media_total
        FROM tru.enem_candidatos
        WHERE ds_escola IS NOT NULL
        GROUP BY ds_escola, sg_uf_residencia, nu_ano
        ORDER BY media_total DESC
    """,
    "enem_renda_desempenho": """
        SELECT
            q006                                    AS faixa_renda,
            nu_ano,
            COUNT(*)                                AS total_candidatos,
            ROUND(AVG(nu_media_total)::numeric, 2)  AS media_total,
            ROUND(AVG(nu_nota_redacao)::numeric, 2) AS media_redacao,
            ROUND(AVG(nu_nota_mt)::numeric, 2)      AS media_mt
        FROM tru.enem_candidatos
        WHERE q006 IS NOT NULL
        GROUP BY q006, nu_ano
        ORDER BY q006
    """
}

for tabela, query in queries.items():
    df = pd.read_sql(query, engine)
    df.to_sql(tabela, engine, schema="ref", if_exists="replace", index=False)
    print(f"✓ ref.{tabela} — {len(df)} linhas")

print("\ntodas as tabelas REF criadas")

In [0]:
COLUNAS_PROJETO = [
    "nu_inscricao", "nu_ano", "tp_sexo",
    "tp_cor_raca", "tp_escola", "tp_faixa_etaria",
    "in_treineiro", "sg_uf_residencia",
    "nu_nota_cn", "nu_nota_ch", "nu_nota_lc", "nu_nota_mt",
    "nu_nota_redacao", "nu_media_total", "q006"
]

buffer2 = io.BytesIO()
df_tru[COLUNAS_PROJETO].to_parquet(buffer2, engine="pyarrow", compression="snappy", index=False)
buffer2.seek(0)

tamanho_mb = buffer2.getbuffer().nbytes / 1024**2
print(f"tamanho: {tamanho_mb:.1f} MB")

if tamanho_mb <= 60:
    s3.put_object(
        Bucket=BUCKET,
        Key="silver/enem/2022/enem_candidatos_projeto.parquet",
        Body=buffer2.getvalue()
    )
    print("parquet do projeto salvo!")
else:
    print("ainda acima de 60MB -> reduzir mais colunas")